# 3-Arms Pipeline with Web Search Context

Two-stage architecture:
- **Stage A**: Tavily web search for all markets (cached to disk)
- **Stage B**: LLM inference with web context (baseline/volume/full-technical)

Run Stage A once, then Stage B can be rerun without wasting Tavily credits.

## Setup: Install Dependencies

In [ ]:
import sys
print(f"Python: {sys.executable}")

# Install required packages
!{sys.executable} -m pip install -U tavily-python openai python-dotenv scikit-learn --quiet

print("\nPackages installed")

In [ ]:
# Imports
import os
import re
import sys
import json
import time
import asyncio
import random
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from tavily import TavilyClient
from openai import AsyncOpenAI

# Load environment
load_dotenv()

# Config
DATA_PATH = "data/markets_microstructure_v2_v3_merged.csv"
CACHE_PATH = "data/tavily_cache.jsonl"
OUT_PATH = "data/output/predictions_3arms_with_web.csv"
MODEL = "llama-3.1-8b-instant"
CONCURRENCY = 8
TAVILY_RPM = 100
TEMPERATURE = 0

os.makedirs("data/output", exist_ok=True)

print(f"Data: {DATA_PATH}")
print(f"Cache: {CACHE_PATH}")
print(f"Output: {OUT_PATH}")
print(f"Model: {MODEL}")
print(f"Concurrency: {CONCURRENCY}")
print(f"Tavily RPM limit: {TAVILY_RPM}")

## Setup: Verify API Keys

In [ ]:
# Check API keys
tavily_key = os.environ.get("TAVILY_API_KEY")
groq_key = os.environ.get("GROQ_API_KEY")

if not tavily_key:
    print("ERROR: TAVILY_API_KEY not found in environment")
    tavily_key = input("Enter TAVILY_API_KEY: ").strip()
else:
    print(f"TAVILY_API_KEY: {tavily_key[:10]}...")

if not groq_key:
    print("ERROR: GROQ_API_KEY not found in environment")
    groq_key = input("Enter GROQ_API_KEY: ").strip()
else:
    print(f"GROQ_API_KEY: {groq_key[:10]}...")

# Initialize clients
tavily_client = TavilyClient(api_key=tavily_key)
groq_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_key,
)

print("\nClients initialized")

## Setup: Smoke Test Tavily

In [ ]:
# Test Tavily search
try:
    test_result = tavily_client.search(
        query="test query",
        search_depth="basic",
        max_results=1
    )
    print("Tavily test successful")
    print(f"Results: {len(test_result.get('results', []))}")
except Exception as e:
    print(f"Tavily test FAILED: {e}")

## Load Markets Data

In [ ]:
# Load markets
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} markets")
print(f"\nRequired columns present:")
for col in ['event_title', 'event_ticker', 'market_ticker', 'mid_yes']:
    print(f"  {col}: {'YES' if col in df.columns else 'MISSING'}")

df.head(2)

---

# STAGE A: Tavily Web Search (Run Once)

Searches all markets and caches results to disk. Supports resume.

In [ ]:
# Stage A: Rate limiter and helper functions

class RateLimiter:
    """Token bucket rate limiter for Tavily API"""
    def __init__(self, rpm):
        self.rpm = rpm
        self.tokens = rpm
        self.last_refill = time.time()
        self.lock = asyncio.Lock()
    
    async def acquire(self):
        async with self.lock:
            # Refill tokens
            now = time.time()
            elapsed = now - self.last_refill
            tokens_to_add = elapsed * (self.rpm / 60.0)
            self.tokens = min(self.rpm, self.tokens + tokens_to_add)
            self.last_refill = now
            
            # Wait if no tokens
            while self.tokens < 1:
                wait_time = (1 - self.tokens) * (60.0 / self.rpm)
                await asyncio.sleep(wait_time)
                now = time.time()
                elapsed = now - self.last_refill
                tokens_to_add = elapsed * (self.rpm / 60.0)
                self.tokens = min(self.rpm, self.tokens + tokens_to_add)
                self.last_refill = now
            
            self.tokens -= 1

def truncate_content(text, max_chars=800):
    """Truncate content to avoid token bloat"""
    if not text:
        return ""
    return text[:max_chars] + ("..." if len(text) > max_chars else "")

def load_cache():
    """Load existing cache and return set of completed market_tickers"""
    if not os.path.exists(CACHE_PATH):
        return set()
    
    completed = set()
    with open(CACHE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line)
                completed.add(record['market_ticker'])
            except:
                pass
    return completed

def append_to_cache(record):
    """Append record to JSONL cache"""
    with open(CACHE_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

async def search_market(row, rate_limiter, semaphore, max_retries=5):
    """Search one market with retry and rate limiting"""
    async with semaphore:
        query = f"{row['event_title']} resolution criteria latest information"
        
        for attempt in range(1, max_retries + 1):
            try:
                # Rate limit
                await rate_limiter.acquire()
                
                # Search
                result = tavily_client.search(
                    query=query,
                    search_depth="basic",
                    max_results=4
                )
                
                # Extract and truncate results
                results = []
                for r in result.get('results', [])[:4]:
                    results.append({
                        'title': r.get('title', '')[:200],
                        'url': r.get('url', '')[:500],
                        'content': truncate_content(r.get('content', ''), 800)
                    })
                
                record = {
                    'market_ticker': row['market_ticker'],
                    'event_ticker': row['event_ticker'],
                    'event_title': row['event_title'],
                    'query': query,
                    'searched_at_utc': datetime.now(timezone.utc).isoformat(),
                    'results': results,
                    'tavily_error': None
                }
                
                append_to_cache(record)
                return True
                
            except Exception as e:
                error_msg = str(e)
                if attempt < max_retries:
                    # Exponential backoff + jitter
                    wait = (2 ** (attempt - 1)) + random.random()
                    await asyncio.sleep(wait)
                else:
                    # Failed after all retries - write error record
                    record = {
                        'market_ticker': row['market_ticker'],
                        'event_ticker': row['event_ticker'],
                        'event_title': row['event_title'],
                        'query': query,
                        'searched_at_utc': datetime.now(timezone.utc).isoformat(),
                        'results': [],
                        'tavily_error': error_msg
                    }
                    append_to_cache(record)
                    return False

print("Stage A functions loaded")

In [ ]:
# Stage A: Run Tavily search for all markets

async def run_stage_a():
    # Load cache and check what's done
    completed = load_cache()
    print(f"Found {len(completed)} already-completed markets in cache")
    
    # Filter to remaining markets
    remaining = df[~df['market_ticker'].isin(completed)]
    print(f"Remaining markets to search: {len(remaining)}")
    
    if len(remaining) == 0:
        print("\nAll markets already cached. Stage A complete.")
        return
    
    # Initialize rate limiter and semaphore
    rate_limiter = RateLimiter(TAVILY_RPM)
    semaphore = asyncio.Semaphore(CONCURRENCY)
    
    print(f"\nStarting Stage A: {len(remaining)} Tavily searches")
    print(f"Rate limit: {TAVILY_RPM} RPM, Concurrency: {CONCURRENCY}")
    print(f"Started: {datetime.now().strftime('%H:%M:%S')}")
    print()
    
    # Create tasks
    tasks = []
    for _, row in remaining.iterrows():
        tasks.append(search_market(row, rate_limiter, semaphore))
    
    # Run with progress tracking
    successful = 0
    for i, task in enumerate(asyncio.as_completed(tasks), 1):
        success = await task
        if success:
            successful += 1
        
        if i % 50 == 0 or i == len(tasks):
            print(f"[{i:4d}/{len(tasks)}] Completed: {successful}/{i} successful ({successful/i*100:.1f}%)")
    
    print(f"\nStage A complete: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Final: {successful}/{len(tasks)} successful")
    print(f"\nCache written to: {CACHE_PATH}")

# Run Stage A
await run_stage_a()

---

# STAGE B: LLM Inference with Web Context

Load cache and run 3-arm forecasting with web search results.

In [ ]:
# Stage B: Load Tavily cache

def load_web_context():
    """Load cache into dict: market_ticker -> formatted web context string"""
    web_context = {}
    
    if not os.path.exists(CACHE_PATH):
        print(f"WARNING: Cache file not found: {CACHE_PATH}")
        print("Run Stage A first to generate web search cache")
        return web_context
    
    with open(CACHE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                record = json.loads(line)
                market_ticker = record['market_ticker']
                
                if record.get('tavily_error'):
                    # Error case - no web context
                    web_context[market_ticker] = "WEB CONTEXT: (search failed)\n"
                elif record.get('results'):
                    # Format results
                    lines = ["WEB CONTEXT:"]
                    for i, r in enumerate(record['results'], 1):
                        lines.append(f"{i}) {r['title']} — {r['url']}")
                        lines.append(f"{r['content']}")
                    web_context[market_ticker] = "\n".join(lines)
                else:
                    web_context[market_ticker] = "WEB CONTEXT: (no results)\n"
            except:
                pass
    
    return web_context

web_context_dict = load_web_context()
print(f"Loaded web context for {len(web_context_dict)} markets")

# Show example
if web_context_dict:
    example_key = list(web_context_dict.keys())[0]
    print(f"\nExample for {example_key}:")
    print(web_context_dict[example_key][:500] + "...")

In [ ]:
# Stage B: Prompt templates with web context

SYSTEM_PROMPT = """You are forecasting the probability this market resolves YES.
Treat mid_yes as a prior probability, then update/reconsider it using your reasoning of the additional signals provided for this branch to bias your output.
Output only one decimal number between 0 and 1 (example: 0.023). No words, no JSON, no punctuation."""

def build_baseline(row, web_context):
    return f"""{web_context}

You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: None (baseline - only question and prior).
Use your reasoning about the question and the web context to update the prior.
Output only the final decimal probability."""

def build_volume(row, web_context):
    # Check which fields are available
    fields = []
    if 'volume' in row.index and pd.notna(row['volume']):
        fields.append(f"- volume: ${row['volume']:,.0f}")
    if 'volume_24h' in row.index and pd.notna(row['volume_24h']):
        fields.append(f"- volume_24h: ${row['volume_24h']:,.0f}")
    if 'open_interest' in row.index and pd.notna(row['open_interest']):
        fields.append(f"- open_interest: ${row['open_interest']:,.0f}")
    if 'liquidity' in row.index and pd.notna(row['liquidity']):
        fields.append(f"- liquidity: ${row['liquidity']:.2f}")
    if 'spread_yes' in row.index and pd.notna(row['spread_yes']):
        fields.append(f"- spread_yes: {row['spread_yes']:.2f}")
    if 'tick_size' in row.index and pd.notna(row['tick_size']):
        fields.append(f"- tick_size: {row['tick_size']:.2f}")
    if 'time_to_close_hours' in row.index and pd.notna(row['time_to_close_hours']):
        fields.append(f"- time_to_close_hours: {row['time_to_close_hours']:.1f}")
    
    fields_text = "\n".join(fields) if fields else "(no participation fields available)"
    
    return f"""{web_context}

You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: volume, volume_24h, open_interest, liquidity, spread_yes, tick_size, time_to_close_hours.
{fields_text}

Update mid_yes using these signals and the web context: higher activity/tighter spread => trust prior more (smaller change); lower activity/wider spread => allow larger reconsideration.
Output only the final decimal probability."""

def build_full_technical(row, web_context):
    # All fields from volume arm plus top-of-book and technical
    fields = []
    
    # Volume/participation
    if 'volume' in row.index and pd.notna(row['volume']):
        fields.append(f"- volume: ${row['volume']:,.0f}")
    if 'volume_24h' in row.index and pd.notna(row['volume_24h']):
        fields.append(f"- volume_24h: ${row['volume_24h']:,.0f}")
    if 'open_interest' in row.index and pd.notna(row['open_interest']):
        fields.append(f"- open_interest: ${row['open_interest']:,.0f}")
    if 'liquidity' in row.index and pd.notna(row['liquidity']):
        fields.append(f"- liquidity: ${row['liquidity']:.2f}")
    
    # Top of book
    if 'yes_bid' in row.index and pd.notna(row['yes_bid']):
        fields.append(f"- yes_bid: {row['yes_bid']:.2f}")
    if 'yes_ask' in row.index and pd.notna(row['yes_ask']):
        fields.append(f"- yes_ask: {row['yes_ask']:.2f}")
    if 'last_price' in row.index and pd.notna(row['last_price']):
        fields.append(f"- last_price: {row['last_price']:.2f}")
    if 'spread_yes' in row.index and pd.notna(row['spread_yes']):
        fields.append(f"- spread_yes: {row['spread_yes']:.2f}")
    if 'tick_size' in row.index and pd.notna(row['tick_size']):
        fields.append(f"- tick_size: {row['tick_size']:.2f}")
    if 'time_to_close_hours' in row.index and pd.notna(row['time_to_close_hours']):
        fields.append(f"- time_to_close_hours: {row['time_to_close_hours']:.1f}")
    
    # Technical indicators
    if 'return_1h' in row.index and pd.notna(row['return_1h']):
        fields.append(f"- return_1h: {row['return_1h']:.3f}")
    if 'return_6h' in row.index and pd.notna(row['return_6h']):
        fields.append(f"- return_6h: {row['return_6h']:.3f}")
    if 'return_24h' in row.index and pd.notna(row['return_24h']):
        fields.append(f"- return_24h: {row['return_24h']:.3f}")
    if 'trend_slope_24h' in row.index and pd.notna(row['trend_slope_24h']):
        fields.append(f"- trend_slope_24h: {row['trend_slope_24h']:.3f}")
    if 'max_drawdown_24h' in row.index and pd.notna(row['max_drawdown_24h']):
        fields.append(f"- max_drawdown_24h: {row['max_drawdown_24h']:.3f}")
    if 'volatility_24h' in row.index and pd.notna(row['volatility_24h']):
        fields.append(f"- volatility_24h: {row['volatility_24h']:.3f}")
    if 'vol_regime_shift' in row.index and pd.notna(row['vol_regime_shift']):
        fields.append(f"- vol_regime_shift: {row['vol_regime_shift']:.3f}")
    if 'high_low_range_24h' in row.index and pd.notna(row['high_low_range_24h']):
        fields.append(f"- high_low_range_24h: {row['high_low_range_24h']:.3f}")
    
    fields_text = "\n".join(fields) if fields else "(no additional fields available)"
    
    return f"""{web_context}

You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: all participation, top-of-book, and technical fields.
{fields_text}

Update mid_yes using these signals and the web context: higher activity/tighter spread => trust prior more (smaller change); lower activity/wider spread => allow larger reconsideration.
Output only the final decimal probability."""

ARMS = {
    'baseline': build_baseline,
    'volume': build_volume,
    'full_technical': build_full_technical,
}

print("Prompt templates loaded with web context integration")

In [ ]:
# Stage B: LLM query functions with retry

def parse_probability(text):
    """Extract probability from response"""
    if not text or not text.strip():
        raise ValueError("Empty response")
    
    # Extract first decimal in [0,1]
    match = re.search(r'([01]?\.\d+|[01])', text.strip())
    if not match:
        raise ValueError(f"No probability in: '{text}'")
    
    p = float(match.group(1))
    if p > 1:
        p = p / 100
    
    if not (0 <= p <= 1):
        raise ValueError(f"Out of range: {p}")
    
    return p

async def query_market_llm(row, arm_name, web_context, semaphore, max_retries=3):
    """Query LLM with retry"""
    async with semaphore:
        # Get web context for this market
        market_ticker = row['market_ticker']
        context = web_context.get(market_ticker, "WEB CONTEXT: (not available)\n")
        
        # Build prompt
        prompt = ARMS[arm_name](row, context)
        
        raw = None
        p_yes = None
        error = None
        
        for attempt in range(1, max_retries + 1):
            try:
                response = await groq_client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=TEMPERATURE,
                )
                raw = response.choices[0].message.content.strip()
                p_yes = parse_probability(raw)
                error = None
                break
            except ValueError as e:
                # Parse error - don't retry
                error = f"Parse: {str(e)}"
                break
            except Exception as e:
                error = f"API: {str(e)}"
                if attempt < max_retries:
                    await asyncio.sleep((2 ** (attempt - 1)) + random.random())
                else:
                    break
        
        return {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'model': MODEL,
            'arm': arm_name,
            'event_ticker': row['event_ticker'],
            'market_ticker': row['market_ticker'],
            'title': row['event_title'],
            'mid_yes': row['mid_yes'],
            'raw_response': raw,
            'p_yes': p_yes,
            'error': error,
            'attempts': attempt,
            'has_web_context': market_ticker in web_context
        }

print("LLM query functions loaded")

In [ ]:
# Stage B: Test with 3 markets

async def test_stage_b():
    sem = asyncio.Semaphore(CONCURRENCY)
    test_markets = df.head(3)
    results = []
    
    print(f"Testing {len(test_markets)} markets x 3 arms = {len(test_markets) * 3} calls\n")
    
    for idx, (_, market) in enumerate(test_markets.iterrows(), 1):
        print(f"[{idx}/3] {market['market_ticker']}: {market['event_title'][:60]}...")
        
        for arm in ['baseline', 'volume', 'full_technical']:
            result = await query_market_llm(market, arm, web_context_dict, sem)
            results.append(result)
            status = "OK" if not result['error'] else f"ERROR: {result['error']}"
            p_str = f"{result['p_yes']:.3f}" if result['p_yes'] else "None"
            web_str = "[+WEB]" if result['has_web_context'] else "[NO WEB]"
            print(f"  {arm:15s}: {p_str} {web_str} - {status}")
    
    df_test = pd.DataFrame(results)
    success = df_test['error'].isna().sum()
    print(f"\nTest: {success}/{len(df_test)} successful")
    
    return df_test

df_test = await test_stage_b()
df_test[['market_ticker', 'arm', 'p_yes', 'error', 'has_web_context']]

In [ ]:
# Stage B: Full pipeline with incremental saves

async def run_stage_b():
    sem = asyncio.Semaphore(CONCURRENCY)
    total_calls = len(df) * 3
    
    # Create tasks
    tasks = []
    for _, row in df.iterrows():
        for arm_name in ARMS.keys():
            tasks.append(query_market_llm(row, arm_name, web_context_dict, sem))
    
    print(f"Starting Stage B: {total_calls} LLM calls ({len(df)} markets x 3 arms)")
    print(f"Started: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Progress updates every 100 calls...\n")
    
    # Process with progress tracking and incremental saves
    results = []
    successful = 0
    
    for i, task in enumerate(asyncio.as_completed(tasks), 1):
        result = await task
        results.append(result)
        
        if result['error'] is None:
            successful += 1
        
        # Incremental save every 500 results
        if i % 500 == 0:
            df_temp = pd.DataFrame(results)
            df_temp.to_csv(OUT_PATH + ".tmp", index=False)
        
        # Progress updates
        if i % 100 == 0 or i == total_calls:
            print(f"[{i:4d}/{total_calls}] Completed: {successful}/{i} successful ({successful/i*100:.1f}%)")
    
    print(f"\nStage B complete: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Final: {successful}/{total_calls} successful ({successful/total_calls*100:.1f}%)")
    
    return pd.DataFrame(results)

df_results = await run_stage_b()

# Final save
df_results.to_csv(OUT_PATH, index=False)
print(f"\nSaved {len(df_results)} predictions to {OUT_PATH}")

In [ ]:
# Summary
total = len(df_results)
success = df_results['error'].isna().sum()
with_web = df_results['has_web_context'].sum()

print(f"Total calls: {total}")
print(f"Successful: {success} ({success/total*100:.1f}%)")
print(f"With web context: {with_web} ({with_web/total*100:.1f}%)")

print("\nBy arm:")
for arm in ['baseline', 'volume', 'full_technical']:
    df_arm = df_results[df_results['arm'] == arm]
    arm_success = df_arm['error'].isna().sum()
    print(f"  {arm:15s}: {arm_success}/{len(df_arm)} ({arm_success/len(df_arm)*100:.1f}%)")

if df_results['error'].notna().sum() > 0:
    print("\nTop errors:")
    print(df_results[df_results['error'].notna()]['error'].value_counts().head(3))

print("\nSample predictions:")
df_results[df_results['error'].isna()].head(6)[['market_ticker', 'arm', 'p_yes', 'has_web_context']]